# 05 - Evaluación del Modelo LDA

**Grupo 15 | UMSS FCyT | IA I/2026**

Objetivos:
- Calcular métricas de coherencia c_v y u_mass
- Comparar distintos valores de K
- Validación cualitativa: ¿los tópicos tienen sentido?
- Inspección de documentos por tópico

In [ ]:
import sys, pickle, warnings
sys.path.append('../')
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import LdaModel, CoherenceModel
from gensim.corpora import Dictionary

from src.preprocesamiento import procesar_corpus
from src.modelo import (
    crear_diccionario_y_corpus, entrenar_lda,
    calcular_coherencia, obtener_topicos, parsear_topico_palabras,
)
sns.set_style('whitegrid')
print('Módulos cargados ✅')

## 1. Cargar modelo guardado (o reentrenar)

In [ ]:
import os

MODEL_PATH = '../data/processed/modelo_lda.model'
DICT_PATH  = '../data/processed/diccionario.dict'
CORP_PATH  = '../data/processed/corpus_procesado.pkl'

if os.path.exists(MODEL_PATH):
    lda_model  = LdaModel.load(MODEL_PATH)
    dictionary = Dictionary.load(DICT_PATH)
    with open(CORP_PATH, 'rb') as f:
        corpus_tokens = pickle.load(f)
    corpus_gensim = [dictionary.doc2bow(d) for d in corpus_tokens]
    print(f'✅ Modelo cargado: K={lda_model.num_topics}')
else:
    # Reentrenar si no existe
    df = pd.read_csv('../data/raw/encuesta_sintetica.csv')
    corpus_tokens = procesar_corpus(df['comentario'].astype(str).tolist())
    corpus_tokens = [[t for t in d if len(t) >= 3] for d in corpus_tokens]
    dictionary, corpus_gensim = crear_diccionario_y_corpus(corpus_tokens)
    lda_model = entrenar_lda(corpus_gensim, dictionary, num_topics=5, passes=20)
    print('✅ Modelo reentrenado con K=5')

## 2. Métricas de coherencia: c_v y u_mass

In [ ]:
# Coherencia c_v (semántica, mayor=mejor)
coh_cv = CoherenceModel(
    model=lda_model, corpus=corpus_gensim,
    dictionary=dictionary, texts=corpus_tokens,
    coherence='c_v'
).get_coherence()

# Coherencia u_mass (estadística, más negativo=mejor)
coh_umass = CoherenceModel(
    model=lda_model, corpus=corpus_gensim,
    dictionary=dictionary,
    coherence='u_mass'
).get_coherence()

print(f'Coherencia c_v    : {coh_cv:.4f}  (rango 0–1, mayor es mejor)')
print(f'Coherencia u_mass : {coh_umass:.4f}  (negativo, más cercano a 0 es mejor)')

## 3. Coherencia por tópico individual

In [ ]:
cm = CoherenceModel(
    model=lda_model, corpus=corpus_gensim,
    dictionary=dictionary, texts=corpus_tokens,
    coherence='c_v'
)
cohs_por_topico = cm.get_coherence_per_topic()

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar([f'T{i}' for i in range(len(cohs_por_topico))],
       cohs_por_topico, color='steelblue')
ax.axhline(coh_cv, color='red', linestyle='--', label=f'Media={coh_cv:.3f}')
ax.set_title('Coherencia c_v por tópico', fontweight='bold')
ax.set_ylabel('Coherencia c_v')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/fig_coherencia_por_topico.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Validación cualitativa de tópicos

In [ ]:
topicos = obtener_topicos(lda_model, num_words=10)

print('VALIDACIÓN CUALITATIVA')
print('=' * 55)
etiquetas_sugeridas = {}
for nombre, palabras_str in topicos.items():
    palabras_dict = parsear_topico_palabras(palabras_str)
    top5 = list(sorted(palabras_dict.items(), key=lambda x: x[1], reverse=True))[:5]
    print(f'\n{nombre}')
    print(f'  Top-5: {[w for w, _ in top5]}')
    print(f'  Interpretación sugerida: [completar manualmente]')

## 5. Documentos representativos por tópico

In [ ]:
df = pd.read_csv('../data/processed/encuesta_con_topicos.csv')

print('COMENTARIOS MÁS REPRESENTATIVOS POR TÓPICO')
print('=' * 55)
for topico_id in sorted(df['topico_dominante'].unique()):
    if topico_id < 0:
        continue
    subset = df[df['topico_dominante'] == topico_id].sort_values('prob_topico', ascending=False)
    print(f'\n🏷️  Tópico {topico_id} (top-3 comentarios):')
    for _, row in subset.head(3).iterrows():
        print(f'  [{row["calificacion_likert"]}★] {row["comentario"][:80]}...')